# Trial run: DeBERTa-v3-base, k = 0, 3 seeds

Temporary pipeline trial (`scripts/trial_run.py`); its numbers are **not reportable** (PROTOCOL §4.1).

Before running, in the notebook settings:
- **Accelerator:** GPU T4
- **Internet:** on (to clone the repo and download the model)
- **Input:** the private `v4_meetings` dataset attached

The repo branch `C/baselines-design` must be pushed: the notebook clones it from GitHub.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
Do **not** install `requirements.txt` here: it would replace Kaggle's own torch build.

In [ ]:
!git clone -b C/baselines-design https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 sentencepiece==0.2.2 protobuf==7.36.2

## 3. Data folder
Found automatically: the folder that contains `source.json`, `train/`, `val/`, `test/`.

In [ ]:
import glob, os

hits = glob.glob("/kaggle/input/**/source.json", recursive=True)
print("source.json found at:", hits)
assert len(hits) == 1, "expected exactly one source.json under /kaggle/input"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "->", sorted(os.listdir(DATA_ROOT)))

## 4. Model loads in fp32
Also downloads the weights once (371 MB).

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/repo/src")
from smi.models.model import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer("microsoft/deberta-v3-base")
print("dtype:", model.dtype)  # must be torch.float32
assert str(model.dtype) == "torch.float32"
print("labels:", model.config.id2label)
del model

## 5. Trial: 3 seeds × 5 epochs

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!PYTHONPATH=src python scripts/trial_run.py --data-root {DATA_ROOT} --model microsoft/deberta-v3-base --out-dir /kaggle/working/runs

## 6. Results

In [ ]:
import json

for run in sorted(glob.glob("/kaggle/working/runs/*")):
    m = json.load(open(f"{run}/metrics.json"))
    print(f"\n=== {os.path.basename(run)} | selected epoch {m['selected_epoch']} | "
          f"macro_f1_6 {m['primary']['macro_f1_6']:.4f} | macro_f1_7 {m['macro_f1_7']:.4f}")
    print("per-class F1:", {c: round(v["f1"], 3) for c, v in m["per_class"].items()})
    for line in open(f"{run}/history.jsonl"):
        h = json.loads(line)
        if h["type"] == "epoch":
            print(f"  epoch {h['epoch']}: train_loss {h['train_loss']:.4f} | macro_f1_6 {h['macro_f1_6']:.4f} "
                  f"| best {h['is_best']} | {h['seconds'] / 60:.1f} min")